# Customer Churn Dataset — Data Cleaning

**Internship submission proof:** This notebook documents the cleaning steps applied to the raw customer churn CSV.

It removes exact duplicate rows, fills missing Region values with `Unknown`, converts `TotalChargesINR` values such as `unknown` to missing numeric values (without inventing billing amounts), and exports the cleaned CSV.

## 1. Import libraries and load the raw dataset
Upload `customer_churn_raw_v2 (1).csv` to the same folder as this notebook, or update `raw_file` below.

In [ ]:
import pandas as pd
import numpy as np

raw_file = 'customer_churn_raw_v2 (1).csv'
df = pd.read_csv(raw_file)

print('Original shape:', df.shape)
display(df.head())

## 2. Inspect missing values and duplicate rows

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
display(df.isna().sum().to_frame('Missing values'))
print("'unknown' TotalChargesINR entries:", df['TotalChargesINR'].astype(str).str.strip().str.lower().eq('unknown').sum())

## 3. Clean the data
- Remove exact duplicate rows.
- Replace missing Region values with `Unknown`.
- Convert `TotalChargesINR` to numeric; non-numeric entries become missing (`NaN`) instead of estimated billing values.
- Convert numeric columns to numeric data types where applicable.

In [ ]:
cleaned = df.copy()

# Remove exact duplicate records
cleaned = cleaned.drop_duplicates().copy()

# Handle missing categorical values
cleaned['Region'] = cleaned['Region'].fillna('Unknown')

# Convert non-numeric/unknown charges to NaN; do not fabricate actual billing values
cleaned['TotalChargesINR'] = pd.to_numeric(
    cleaned['TotalChargesINR'].replace(r'^\\s*unknown\\s*$', np.nan, regex=True),
    errors='coerce'
)

# Ensure numeric columns have numeric types
numeric_cols = ['Age', 'TenureMonths', 'MonthlyChargesINR', 'TotalChargesINR',
                'SupportTickets', 'LatePayments']
for col in numeric_cols:
    cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')

print('Cleaned shape:', cleaned.shape)
display(cleaned.head())

## 4. Validate the cleaned dataset

In [ ]:
print('Remaining duplicate rows:', cleaned.duplicated().sum())
print('Missing values by column:')
display(cleaned.isna().sum().to_frame('Missing values'))

assert cleaned.duplicated().sum() == 0
assert cleaned.shape[1] == df.shape[1]
print('Validation complete: no duplicate rows; all original columns retained.')

## 5. Export cleaned CSV

In [ ]:
output_file = 'customer_churn_cleaned_final.csv'
cleaned.to_csv(output_file, index=False)
print(f'Cleaned dataset saved as: {output_file}')

## Cleaning summary

- Raw records: 605
- Exact duplicate rows removed: 5
- Final records: 600
- Original columns retained: 13
- Missing Region values were labeled `Unknown`.
- `unknown` TotalChargesINR entries were converted to missing numeric values; no estimated billing values were fabricated.

**Note:** A remaining missing numeric value is intentionally preserved when the source does not provide a reliable amount. Handle it separately according to the analysis/modeling requirement.